In [ ]:
# ---
# Convert RefSeq RNA IDs (XM_, XR_) to PmUG01 gene IDs
# Uses NCBI Entrez API
# Prioritizes PmUG01 locus tags, then synonyms, then gene symbols
# Saves progress after each batch to prevent data loss
# ---


In [ ]:

import pandas as pd
import time
from Bio import Entrez


# Configuration
Entrez.email = "halsal4u@gmail.com"  # Required by NCBI
input_file = "analysis_results/variants_analysis/files/cleaned_extracted_variants.tsv"
output_file = "analysis_results/variants_analysis/files/PmUGO_converted_genes.csv"
BATCH_SIZE = 10

# Load Data & Filter
df = pd.read_csv(input_file, sep=None, engine="python")
genes_to_convert = [
    row["Gene"]
    for _, row in df.iterrows()
    if isinstance(row["Gene"], str) and row["Gene"].startswith(("XM_", "XR_")) and
    (str(row.get("PmUG0_ID", "")).strip() == "" or not str(row["PmUG0_ID"]).startswith(("PmUG01", "PMUG01")))
]

# Fetch PmUG01 ID from NCBI
def fetch_pmug_id(refseq_id):
    """Retrieve PmUG01 gene ID or symbol using RefSeq RNA accession."""
    try:
        query = f"{refseq_id}[srcdb_refseq]"
        print(f"🔍 Searching NCBI for: {query}")

        # Step 1: Search gene ID
        handle = Entrez.esearch(db="gene", term=query, retmode="xml")
        record = Entrez.read(handle)
        handle.close()

        if not record["IdList"]:
            print(f"⚠ No match found for {refseq_id}")
            return "Not Found"

        gene_id = record["IdList"][0]

        # Step 2: Fetch gene summary
        handle = Entrez.efetch(db="gene", id=gene_id, retmode="xml")
        gene_data = Entrez.read(handle)
        handle.close()

        # Step 3: Try multiple strategies to find PmUG01 ID
        gene_info = gene_data[0].get("Entrezgene_gene", {}).get("Gene-ref", {})
        pmug_id = gene_info.get("Gene-ref_locus-tag", "Not Found")

        if pmug_id == "Not Found":
            synonyms = gene_info.get("Gene-ref_syn", [])
            pmug_id = next((syn for syn in synonyms if syn.startswith("PmUG01")), "Not Found")

        if pmug_id == "Not Found":
            gene_source = gene_data[0].get("Entrezgene_gene-source", {})
            pmug_id = gene_source.get("Gene-source_src-str2", "Not Found")

        if pmug_id == "Not Found":
            pmug_id = gene_info.get("Gene-ref_locus", "Not Found")

        print(f"✅ {refseq_id} → {pmug_id}")
        return pmug_id

    except Exception as e:
        print(f"❌ Error fetching {refseq_id}: {e}")
        return "Not Found"


# Batch Process & Update
converted_results = {}

for i in range(0, len(genes_to_convert), BATCH_SIZE):
    batch = genes_to_convert[i:i + BATCH_SIZE]
    print(f"\n🔄 Processing batch {i // BATCH_SIZE + 1}/{-(-len(genes_to_convert) // BATCH_SIZE)}...")

    for refseq_id in batch:
        converted_results[refseq_id] = fetch_pmug_id(refseq_id)
        time.sleep(0.5)  # Respect NCBI rate limits

    # Update DataFrame with current batch results
    df["PmUG0_ID"] = df.apply(
        lambda row: converted_results.get(row["Gene"], row["PmUG0_ID"]),
        axis=1
    )

    df.to_csv(output_file, index=False)
    print("✅ Progress saved.")

# Final Output
print(f"\n🎉 Conversion complete! Results saved to '{output_file}'")
